# Minor Project 2 - Predict New York City Airbnb Prices
**Linear Regression Project | AIML Batch**

This notebook follows the seven steps given in the project brief. The aim is to build a simple and explainable Linear Regression model for NYC Airbnb nightly prices.

## Step 1 - Exploratory Data Analysis (EDA) and Data Auditing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# New York City Airbnb Open Data (AB_NYC_2019)
# Keep AB_NYC_2019.csv in the same folder. In Colab, upload it before running.
data = pd.read_csv("https://raw.githubusercontent.com/YueminLi/Airbnb_NYC_2019/master/AB_NYC_2019.csv")

print('Dataset shape:', data.shape)
print('\nColumns:')
print(data.columns.tolist())
print('\nData types:')
print(data.dtypes)
data.head()


In [ ]:
# Basic numerical and categorical separation
numerical_cols = data.select_dtypes(include=np.number).columns.tolist()
categorical_cols = data.select_dtypes(exclude=np.number).columns.tolist()
print('Numerical columns:', numerical_cols)
print('Categorical columns:', categorical_cols)

# Task 1.2 - classify the variables more specifically
discrete_numerical = ['id', 'host_id', 'price', 'minimum_nights', 'number_of_reviews', 'calculated_host_listings_count', 'availability_365']
continuous_numerical = ['latitude', 'longitude', 'reviews_per_month']
nominal_categorical = ['name', 'host_name', 'neighbourhood_group', 'neighbourhood', 'room_type', 'last_review']
ordinal_categorical = []  # no main categorical field has a natural ranking
print('\nDiscrete numerical:', discrete_numerical)
print('Continuous numerical:', continuous_numerical)
print('Nominal categorical:', nominal_categorical)
print('Ordinal categorical:', ordinal_categorical)

# Missing-value audit with percentages
missing = pd.DataFrame({
    'missing_count': data.isnull().sum(),
    'missing_percent': (data.isnull().mean() * 100).round(2)
})
missing[missing.missing_count > 0]


**Audit note.** The dataset has about 49,000 listings. `last_review` and `reviews_per_month` contain substantial missing data. `name` and `host_name` can also contain a few blanks, but they are not useful predictors for this model.

## Step 2 - Data Cleaning and Missing Values

In [ ]:
df = data.copy()

# If a listing has no reviews, a missing reviews_per_month logically means zero review activity.
print(pd.crosstab(df["reviews_per_month"].isna(), df["number_of_reviews"].eq(0)))
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# last_review has many missing dates and is not being engineered for this basic model.
# Unique identifiers and names do not give a generalisable mathematical signal.
drop_cols = ["id", "name", "host_id", "host_name", "last_review"]
df = df.drop(columns=drop_cols)

print("Remaining missing values:", int(df.isnull().sum().sum()))
print("Shape after basic cleaning:", df.shape)

## Step 3 - Outlier Identification and Treatment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df["price"], ax=axes[0])
axes[0].set_title("Price before outlier treatment")
sns.boxplot(x=df["minimum_nights"], ax=axes[1])
axes[1].set_title("Minimum nights before outlier treatment")
plt.tight_layout()
plt.show()

price_95 = df["price"].quantile(0.95)
nights_95 = df["minimum_nights"].quantile(0.95)
print("95th percentile - price:", price_95)
print("95th percentile - minimum nights:", nights_95)

rows_before = len(df)
df = df[(df["price"] > 0) & (df["price"] <= price_95) &
        (df["minimum_nights"] <= nights_95)].copy()
print("Rows before trimming:", rows_before)
print("Rows after trimming:", len(df))

I used trimming rather than changing the extreme values. This keeps the model focused on ordinary consumer listings and reduces the influence of very expensive listings and unusually long minimum stays.

## Step 4 - Feature Selection and Categorical Encoding

In [ ]:
print("Unique neighbourhoods:", df["neighbourhood"].nunique())
print("Unique neighbourhood groups:", df["neighbourhood_group"].nunique())

# More than 200 neighbourhood labels would create a very wide set of dummy variables.
# For this introductory model, the five borough-level groups are easier to interpret.
df = df.drop(columns=["neighbourhood"])

# One-hot encoding; drop_first=True avoids redundant dummy columns.
df_encoded = pd.get_dummies(
    df, columns=["neighbourhood_group", "room_type"], drop_first=True, dtype=int
)
print("Encoded shape:", df_encoded.shape)
df_encoded.head()

## Step 5 - Feature/Target Separation and 80/20 Split

In [ ]:
X = df_encoded.drop(columns="price")
y = df_encoded["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Step 6 - Linear Regression Training and Evaluation

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared: {r2:.4f}")
print(f"MAE: ${mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: ${rmse:.2f}")

**How to read the metrics:** R² is the share of price variation explained by the model. MAE is the average absolute dollar difference between the prediction and the real nightly price. RMSE gives extra weight to larger mistakes.

## Step 7 - Residual Analysis and Interpretation

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.25)
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
plt.plot([low, high], [low, high], "--")
plt.xlabel("Actual price ($)")
plt.ylabel("Predicted price ($)")
plt.title("Actual vs Predicted Airbnb Prices")
plt.show()

residuals = y_test - y_pred
plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.25)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted price ($)")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residual Plot")
plt.show()

In [ ]:
# Compare absolute errors for cheaper and more expensive listings.
results = pd.DataFrame({"actual": y_test, "predicted": y_pred})
results["absolute_error"] = (results["actual"] - results["predicted"]).abs()
price_split = y_test.median()
results["segment"] = np.where(results["actual"] <= price_split, "Cheaper listings", "Expensive listings")
segment_error = results.groupby("segment")["absolute_error"].mean().round(2)
print("Test-set median price: $", round(price_split, 2))
print(segment_error)

### Final project answer
The model generally performs better on cheaper listings. The lower-price market is more concentrated around common room types and borough-level patterns. Expensive listings have more variation caused by factors that are not captured in this dataset/model, such as property quality, exact micro-location, amenities, view, interior condition and special events. Linear Regression also tends to pull extreme predictions toward the average, so errors become larger at the expensive end.

## Conclusion
This project shows the complete regression workflow: auditing the raw data, making justified cleaning decisions, controlling outliers, encoding categories, creating a reproducible train/test split, fitting Linear Regression, evaluating it with R²/MAE/RMSE, and checking residual behaviour. The model is useful as a baseline, but its errors also show why richer property features or more flexible models could improve price prediction.